# inbox_app

> Keyboard-driven TUI for triaging the Braincave inbox. Stage 1 of the human-gated pipeline.

Runs fully offline — no LLM, no network, and no need for the Obsidian desktop app except for
the optional `o` (open) key. That is the whole point: the two moments Jelle touches a note are
pure human judgment, and the Hopsakee.Decimal proposals happen overnight in between.

```
uv run python -m filesorter.inbox_app            # whole inbox, in processing order
uv run python -m filesorter.inbox_app --folder Phone --limit 60
```

| Key | Verdict |
|-----|---------|
| `h` | keep — pick a `note-status`, a domain, and optionally an area. Writes `note-status:` + `hd:`, no move |
| `d` | delete — *queued* for `obsidian delete` (OS trash); the file is never touched here |
| `t` | too hard — appends `- [ ] Organise this note [15m] ⏳ <tomorrow>` to the note |
| `s` | skip |
| `u` | undo — restores the previous note bytes exactly |
| `o` | open in Obsidian |
| `q` | quit |

In [ ]:
#| default_exp inbox_app

In [ ]:
#| export
from pathlib import Path

from rich.text import Text
from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Horizontal, Vertical
from textual.screen import ModalScreen
from textual.widgets import Footer, Header, Label, ListItem, ListView, Static

from filesorter.inbox import (
    INBOX,
    HDArea,
    HDDomain,
    Triage,
    backlink_count,
    build_backlink_index,
    hd_structure_age_days,
    list_candidates,
    load_hd_domains,
    load_note_statuses,
    note_preview,
    pending_deletes,
    obsidian_open,
    parse_note,
)

## Pickers

Every row carries its own shortcut key, so a keeper costs four keystrokes (`h`, status,
domain, area) and arrow-key browsing stays available for rows whose position Jelle has not
memorised.

The area picker is the nice one: its shortcut key *is* the digit that gets written. Pressing
`7` in `d6` stores `hd: d6.7`, which is a truncated `d6.75` — so an area needs no syntax of
its own. Space stores the bare domain when the area is not obvious.

In [ ]:
#| export
class _KeyedPicker(ModalScreen[str | None]):
    """Modal list where each row has its own shortcut key. Esc cancels."""

    BINDINGS = [Binding("escape", "cancel", "Cancel")]
    title_text = "SELECT"
    hint_text = "Press a key or browse · [ESC] Cancel"

    def __init__(self, options: list[tuple[str, str, str]]):
        """`options` is a list of `(value, label, shortcut_key)` triples."""
        super().__init__()
        self.options = options

    def compose(self) -> ComposeResult:
        yield Vertical(
            Label(self.title_text, id="modal-title"),
            ListView(*[
                ListItem(Label(f"  {key if key != 'space' else '␣'}  {label}"), id=f"opt-{i}")
                for i, (_, label, key) in enumerate(self.options)
            ], id="opt-list"),
            Label(f"  {self.hint_text}", id="modal-hint"),
            id="modal-container",
        )

    def on_list_view_selected(self, event: ListView.Selected) -> None:
        if event.item.id and event.item.id.startswith("opt-"):
            self.dismiss(self.options[int(event.item.id[4:])][0])

    def on_key(self, event) -> None:
        for value, _, key in self.options:
            if event.key == key:
                event.stop()
                self.dismiss(value)
                return

    def action_cancel(self) -> None:
        self.dismiss(None)


class StatusPicker(_KeyedPicker):
    """Pick a `note-status`. Values come from the vault, never from this code."""
    title_text = "NOTE-STATUS — your call, not Hoggle's"


class DomainPicker(_KeyedPicker):
    """Pick the Hopsakee.Decimal domain."""
    title_text = "DOMAIN"


class AreaPicker(_KeyedPicker):
    """Pick an area — optional, and its shortcut key IS the digit that gets written.

    Area `70-79` is reached with `7` and stores `hd: d6.7`, which is just a truncated
    `d6.75`. Space stores the domain alone when the area is not obvious.
    """
    title_text = "AREA — optional, narrows what the proposer has to guess"
    hint_text = "Press the area digit · [␣] domain only · [ESC] Cancel"


#: Sentinel returned by AreaPicker when Jelle declines to narrow to an area.
DOMAIN_ONLY = "__domain_only__"

## Panels

The left panel carries the decision-relevant metadata — including the inbound-link count,
which is the signal that stops a connected note from being deleted by reflex.

In [ ]:
#| export
class NoteInfoPanel(Static):
    """Left panel: which note, where from, how connected."""

    def update_note(self, path: Path | None, index: int, total: int, backlinks: int, header: list[str]):
        if path is None:
            self.update(Text("Inbox empty for this selection.", style="bold green"))
            return
        try: folder = path.relative_to(INBOX).parts[0]
        except ValueError: folder = path.parent.name

        text = Text()
        text.append(f"Note {index + 1} of {total}\n\n", style="bold cyan")
        text.append(f"{path.stem}\n", style="bold white")
        text.append(f"{folder}\n", style="dim")
        style = "bold yellow" if backlinks else "dim"
        text.append(f"\n{backlinks} inbound link{'' if backlinks == 1 else 's'}", style=style)
        if backlinks: text.append("  ← connected", style="bold yellow")
        text.append("\n")
        if header:
            text.append("\n")
            for line in header: text.append(f"{line}\n", style="green")
        self.update(text)


class NotePreviewPanel(Static):
    """Right panel: the note's actual body, frontmatter stripped."""

    def update_preview(self, body: list[str] | None):
        if body is None: self.update("")
        elif not body: self.update(Text("(no body — frontmatter only)", style="dim"))
        else: self.update("\n".join(body))

In [ ]:
#| export
APP_CSS = """
Screen { background: $surface; }
#main-panels { height: 1fr; }
#note-info { width: 1fr; height: 100%; padding: 1 2; border-right: solid $primary; }
#preview { width: 2fr; padding: 1 2; overflow-y: auto; }
#status-bar { dock: bottom; height: 3; padding: 0 2; background: $boost; }
#modal-container { width: 78; height: 80%; border: thick $primary; background: $surface; padding: 1 2; }
#modal-title { text-style: bold; color: $accent; padding: 0 0 1 0; }
#modal-hint { color: $text-muted; padding: 1 0 0 0; }
StatusPicker, DomainPicker, AreaPicker { align: center middle; }
"""

## The app

`keep` is the only multi-step verdict: status, then domain. Both are Jelle's choice — the app
never pre-selects or guesses either.

In [ ]:
#| export
class InboxTriageApp(App):
    """Triage the Braincave inbox one note at a time."""

    CSS = APP_CSS
    TITLE = "Braincave Inbox Triage"
    BINDINGS = [
        Binding("h", "keep", "Keep"),
        Binding("d", "delete", "Delete"),
        Binding("t", "todo", "Too hard"),
        Binding("s", "skip", "Skip"),
        Binding("u", "undo", "Undo"),
        Binding("o", "open_note", "Open"),
        Binding("q", "quit_app", "Quit"),
    ]

    def __init__(self, folder: str | None = None, limit: int | None = None):
        super().__init__()
        self.folder = folder
        self.limit = limit
        self.triage = Triage()
        self.statuses: list[str] = []
        self.domains: list[HDDomain] = []
        self.backlinks: dict[str, int] = {}
        self.notes: list[Path] = []
        self.index = 0

    # ---- lifecycle -------------------------------------------------------

    def on_mount(self) -> None:
        self.statuses = load_note_statuses()
        self.domains = load_hd_domains()
        self.backlinks = build_backlink_index()
        self._reload_notes()
        self._show_current()
        age = hd_structure_age_days()
        if age > 14:
            self._status(f"⚠ hd-structure.json is {age:.0f} days old — run _HD BuildCache")
        elif (queued := len(pending_deletes())):
            self._status(f"{queued} note(s) already queued for delete — review before draining")

    def _reload_notes(self) -> None:
        notes = list_candidates()
        if self.folder:
            wanted = self.folder.lower()
            notes = [p for p in notes if p.relative_to(INBOX).parts[0].lower() == wanted]
        self.notes = notes[:self.limit] if self.limit else notes

    @property
    def current(self) -> Path | None:
        return self.notes[self.index] if 0 <= self.index < len(self.notes) else None

    def _show_current(self) -> None:
        path = self.current
        info = self.query_one("#note-info", NoteInfoPanel)
        preview = self.query_one("#preview", NotePreviewPanel)
        if path is None:
            info.update_note(None, self.index, len(self.notes), 0, [])
            preview.update_preview(None)
            self._status(f"Done — {self.triage.summary()}")
            return
        header, body = note_preview(parse_note(path))
        info.update_note(path, self.index, len(self.notes), backlink_count(self.backlinks, path), header)
        preview.update_preview(body)
        self.sub_title = str(path.parent)

    def _status(self, message: str) -> None:
        self.query_one("#status-bar", Static).update(Text(message))

    def _advance(self, message: str) -> None:
        """Report the verdict, then move on. Triaged notes stay in the list until reload."""
        self.index += 1
        self._show_current()
        self._status(message)

    def compose(self) -> ComposeResult:
        yield Header()
        with Horizontal(id="main-panels"):
            yield NoteInfoPanel(id="note-info")
            yield NotePreviewPanel(id="preview")
        yield Static(id="status-bar")
        yield Footer()

    # ---- verdicts --------------------------------------------------------

    def action_keep(self) -> None:
        """Keep this note: Jelle picks a status, a domain, and optionally an area."""
        if self.current is None: return
        self.push_screen(
            StatusPicker([(s, s, str(i + 1)) for i, s in enumerate(self.statuses)]),
            self._after_status,
        )

    def _after_status(self, status: str | None) -> None:
        if status is None:
            self._status("Cancelled")
            return
        self.push_screen(
            DomainPicker([(d.id, d.label, str(i + 1)) for i, d in enumerate(self.domains)]),
            lambda domain_id: self._after_domain(status, domain_id),
        )

    def _after_domain(self, status: str, domain_id: str | None) -> None:
        if domain_id is None:
            self._status("Cancelled")
            return
        domain = next((d for d in self.domains if d.id == domain_id), None)
        if domain is None:
            self._status("Cancelled")
            return
        options = [(a.id, a.label, a.hd_digit) for a in domain.areas]
        options.append((DOMAIN_ONLY, f"— no area, file as {domain.id} —", "space"))
        self.push_screen(
            AreaPicker(options),
            lambda area_id: self._after_area(status, domain, area_id),
        )

    def _after_area(self, status: str, domain, area_id: str | None) -> None:
        path = self.current
        if area_id is None or path is None:
            self._status("Cancelled")
            return
        area: HDArea | None = None
        if area_id != DOMAIN_ONLY:
            area = next((a for a in domain.areas if a.id == area_id), None)
        self._advance(f"✓ {path.stem} → {self.triage.keep(path, status, domain.hd(area))}")

    def action_delete(self) -> None:
        path = self.current
        if path is None: return
        self._advance(f"🗑 {path.stem} → {self.triage.delete(path)}")

    def action_todo(self) -> None:
        path = self.current
        if path is None: return
        self._advance(f"📅 {path.stem} → {self.triage.todo(path)}")

    def action_skip(self) -> None:
        path = self.current
        if path is None: return
        self._advance(f"⏭ skipped {path.stem}")

    def action_undo(self) -> None:
        if not self.triage.can_undo():
            self._status("⚠ Nothing to undo")
            return
        action = self.triage.undo()
        if action is None: return
        try: self.index = self.notes.index(action.note)
        except ValueError: self.index = max(0, self.index - 1)
        self._show_current()
        self._status(f"↩ undone: {action.kind} on {action.note.stem}")

    def action_open_note(self) -> None:
        path = self.current
        if path is None: return
        try:
            res = obsidian_open(path)
            ok = res.returncode == 0
            self._status(f"📖 opened {path.stem}" if ok else "⚠ obsidian CLI failed — is the app running?")
        except Exception as e:
            self._status(f"⚠ could not open: {e}")

    def action_quit_app(self) -> None:
        """Quit, always reporting the delete queue so nothing accumulates unseen."""
        parts = [self.triage.summary()]
        if (queued := len(pending_deletes())):
            parts.append(f"{queued} note(s) queued for delete — nothing removed yet; "
                         f"run drain_deletes(dry_run=False) to apply")
        self.exit(message=" | ".join(parts))

In [ ]:
#| export
def main():
    """CLI entry point for inbox triage."""
    import argparse

    parser = argparse.ArgumentParser(description="Triage the Braincave inbox")
    parser.add_argument("--folder", default=None,
                        help="Restrict to one inbox subfolder, e.g. Phone")
    parser.add_argument("--limit", type=int, default=None,
                        help="Stop after N notes — handy for a 15-minute session")
    args = parser.parse_args()
    result = InboxTriageApp(folder=args.folder, limit=args.limit).run()
    if result: print(result)


if __name__ == "__main__":
    main()

## Smoke test

The app itself needs a terminal, so the notebook only checks that it constructs and that its
data contracts hold.

In [ ]:
app = InboxTriageApp(folder="Phone", limit=5)
print(sorted(b.key for b in app.BINDINGS))
assert {b.key for b in app.BINDINGS} == {"h", "d", "t", "s", "u", "o", "q"}
assert load_note_statuses(), "status vocabulary must be readable"

doms = load_hd_domains()
assert all(d.id != "d0" for d in doms)

# The area shortcut key must equal the digit written into `hd:`.
for d in doms:
    opts = [(a.id, a.label, a.hd_digit) for a in d.areas]
    assert len({k for _, _, k in opts}) == len(opts), f"{d.id} has duplicate area shortcuts"
    for a in d.areas:
        assert d.hd(a) == f"{d.id}.{a.hd_digit}"
d6 = next(d for d in doms if d.id == "d6")
a70 = next(a for a in d6.areas if a.id == "70-79")
print("d6 70-79 →", d6.hd(a70), "| domain only →", d6.hd())
assert d6.hd(a70) == "d6.7" and d6.hd() == "d6"
assert "d6.75".startswith(d6.hd(a70)), "area value must be a prefix of its categories"
print("app constructs; area digits are unique per domain and prefix their categories")